# E4 — Reach sweep

`e4_reach.py`. e_F against the position along the line, and RIM's Λ eigenvalues there. Expected: M4 is
right at its tuning point (the start of the line) and degrades away from it; M5 does not.

In [ ]:
# Runs to analyse: run directories, session files (exp/<exp>/sessions/*.txt, written by the script) or a
# folder of runs. Default: every e4 run on the robot.
RUNS = ["~/data/franka/rim-delays/robot/e4"]
RECOMPUTE = False  # True: recompute the metrics from the data instead of reading analysis/metrics.json

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()  # experiments/rim-delays/exp
sys.path[:0] = [str(HERE), str(HERE.parent)]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import exp_metrics
import plots as P
from common import collect, load_table

%matplotlib inline
pd.set_option("display.precision", 3)

runs = collect(RUNS)
df = load_table(runs, exp_metrics.e4 if RECOMPUTE else None)
print(f"{len(df)} runs")

In [ ]:
df[["run", "method", "rtt", "line_length_m", "eF_rms", "E_end_J"]].sort_values(["rtt", "method"])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
for _, row in df.iterrows():
    m = P.metrics_of(row.dir)
    y = np.array([np.nan if v is None else v for v in m["eF_rms_by_s"]])
    axes[0].plot(m["s_m"], y, marker="o", color=P.color(row.method), ls="-" if P.delay_key(row.rtt) == "0" else "--",
                 label=f"{P.LABELS.get(row.method, row.method)}, T = {P.delay_key(row.rtt)}")
    if "lambda_eig_by_s" in m:
        eig = np.array([[np.nan] * 3 if v is None else v for v in m["lambda_eig_by_s"]])
        for k, name in enumerate(("lightest", "middle", "heaviest")):
            axes[1].plot(m["s_m"], eig[:, k], color=["#2a78d6", "#eb6834", "#1baf7a"][k], label=name)
axes[0].set_ylabel("RMS e_F [N]")
axes[1].set_ylabel("Λ eigenvalue [kg]")
axes[1].set_xlabel("position along the line [m]")
axes[0].legend(fontsize=8)
handles, labels = axes[1].get_legend_handles_labels()
unique = dict(zip(labels, handles))  # one entry per eigenvalue, not per run
axes[1].legend(unique.values(), unique.keys(), fontsize=8)
fig.tight_layout()